In [1]:
import sys
import os
import django
import json
import pandas as pd
import gc
import torch
from concurrent.futures import ProcessPoolExecutor, as_completed

# ---------------------------
# Settings
# ---------------------------
project_path = "/home/pdapp/pd_api_server/"
if project_path not in sys.path:
    sys.path.insert(0, project_path)

# Allow Django ORM in Jupyter async environment
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "mysite.settings")
django.setup()

from api.models import Patient

# Paths
JSON_PATH = "./20260316_prediction.json"
CSV_PATH = "./20260316_prediction.csv"

# Parallel workers
MAX_WORKERS = 3  # Adjust depending on CPU/GPU

In [2]:
# ---------------------------
# Worker function
# ---------------------------
def run_single_patient(patient_id, name):
    """
    Worker function to run prediction for a single patient.
    Returns: name, result list, error (None if success)
    """
    import os
    import django
    import gc
    import torch

    os.environ.setdefault("DJANGO_SETTINGS_MODULE", "mysite.settings")
    django.setup()

    from api.views import run_predict_model

    try:
        # Run model
        result = run_predict_model(patient_id, save_results=False)

        output = [
            result["gait_result"],
            result["voice_result"],
            result["hand_result"],
            result["multimodal_results"]
        ]

        return name, output, None

    except Exception as e:
        return name, None, str(e)

    finally:
        # Force memory cleanup
        gc.collect()
        torch.cuda.empty_cache()

In [3]:
# ---------------------------
# Load existing progress
# ---------------------------
if os.path.exists(JSON_PATH):
    with open(JSON_PATH, "r") as f:
        result_dt = json.load(f)
else:
    result_dt = {}

failed = []

# ---------------------------
# Fetch patients
# ---------------------------
patients = list(Patient.objects.all())

# Prepare tasks for patients not yet processed
tasks = [(p.patientId, p.name) for p in patients if p.name not in result_dt]

print("Total remaining patients:", len(tasks))

Total remaining patients: 472


In [4]:
# ---------------------------
# Parallel Execution
# ---------------------------
if __name__ == "__main__":
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:

        futures = [
            executor.submit(run_single_patient, pid, name)
            for pid, name in tasks
        ]

        for future in as_completed(futures):
            name, result, error = future.result()

            if error:
                print(f"Failed: {name} -> {error}")
                failed.append((name, error))
            else:
                print(f"Finished: {name}")
                result_dt[name] = result

                # Save progress immediately
                with open(JSON_PATH, "w") as f:
                    json.dump(result_dt, f, indent=2)

/home/pdapp/miniconda3/envs/pdapp/lib/python3.8/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
/home/pdapp/miniconda3/envs/pdapp/lib/python3.8/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
/home/pdapp/miniconda3/envs/pdapp/lib/python3.8/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 

/mnt/pd_app/results/ /results/
[<class 'rest_framework_simplejwt.authentication.JWTAuthentication'>] [<class 'rest_framework.permissions.IsAuthenticated'>]
Read patient object: Age 80, gender 1, name 吳克洋
/mnt/pd_app/results/ /results/
[<class 'rest_framework_simplejwt.authentication.JWTAuthentication'>] [<class 'rest_framework.permissions.IsAuthenticated'>]
Read patient object: Age 70, gender 1, name Leo
/mnt/pd_app/results/ /results/
[<class 'rest_framework_simplejwt.authentication.JWTAuthentication'>] [<class 'rest_framework.permissions.IsAuthenticated'>]
Read patient object: Age 34, gender 1, name 范淞斌
Failed: 吳克洋 -> Missing required files
Read patient object: Age 24, gender 1, name 冼冠宇
Failed: 范淞斌 -> Missing required files
Failed: 冼冠宇 -> Missing required filesRead patient object: Age 0, gender 1, name 0313testadd

Read patient object: Age 81, gender 1, name 04_2025-313–05
Failed: 0313testadd -> Missing required files
Read patient object: Age 81, gender 1, name 004-20250313-05
Failed

/home/pdapp/miniconda3/envs/pdapp/lib/python3.8/site-packages/numpy/core/fromnumeric.py:3432: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/pdapp/miniconda3/envs/pdapp/lib/python3.8/site-packages/numpy/core/_methods.py:190: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


Failed: 004-20250313_07 -> Input X contains NaN.
RandomForestClassifier does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values
Read patient object: Age 24, gender 1, name 冼冠宇
Failed: 冼冠宇 -> Missing required files
Read patient object: Age 71, gender 2, name 20250505_110713


/home/pdapp/miniconda3/envs/pdapp/lib/python3.8/site-packages/numpy/core/fromnumeric.py:3432: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/pdapp/miniconda3/envs/pdapp/lib/python3.8/site-packages/numpy/core/_methods.py:190: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


...

Failed: 20260311_092804 -> Input X contains NaN.
RandomForestClassifier does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values
Read patient object: Age 83, gender 2, name 20260311_101814
finish models prediction
Finished: 20260311_094353
Read patient object: Age 63, gender 2, name 20260311_105557
finish models prediction
Finished: 20260311_095828
Read patient object: Age 70, gender 2, name 20260313_091606


/home/pdapp/pd_api_server/api/pdModel/handFeaturesExtraction.py:239: RuntimeWarning: invalid value encountered in divide
  _last_half_average_frequency = np.mean(np.sum(last_msg * f[:, None], axis=0) / np.sum(last_msg, axis=0))


Failed: 20260311_101814 -> Input X contains NaN.
RandomForestClassifier does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values
Read patient object: Age 85, gender 1, name 20260313_093448
finish models prediction
Finished: 20260311_105557
Read patient object: Age 78, gender 2, name 20260313_095147
finish models prediction
Finished: 20260313_091606
Read patient object: Age 65, gender 2, name 20260313_101322
finish models prediction
Finished: 20260313_09

/home/pdapp/pd_api_server/api/pdModel/handFeaturesExtraction.py:239: RuntimeWarning: invalid value encountered in divide
  _last_half_average_frequency = np.mean(np.sum(last_msg * f[:, None], axis=0) / np.sum(last_msg, axis=0))


finish models prediction
Finished: 20260313_133141
Read patient object: Age 81, gender 1, name 20260313_142803
Failed: 20260313_135000 -> Input X contains NaN.
RandomForestClassifier does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-valuesRead patient object: Age 71, gender 1, name 20260316_140615

finish models prediction
Finished: 20260313_141344
Read patient object: Age 69, gender 2, name 20260316_142500
finish models prediction
Finished: 20260313_14

In [5]:
# # ---------------------------
# # Convert JSON to CSV
# # ---------------------------
# with open(JSON_PATH, "r") as f:
#     data = json.load(f)

# df = pd.DataFrame.from_dict(data).T
# df.columns = ["gait", "voice", "hand", "multimodal"]
# df.to_csv(CSV_PATH)

# print("CSV saved:", CSV_PATH)
# print(f"Total failed patients: {len(failed)}")